# Module 8.4 — Orientation: Web Frameworks

### Python Mastery · Track 8: Real-World Python

Web frameworks turn Python into the engine behind websites and APIs. This module is an **orientation**, not a full web-development course: it explains the request/response model, the WSGI and ASGI standards, and the two most common frameworks, Flask and FastAPI, so you know what they are and when to reach for each. Full web development is its own large subject; this points you in the right direction.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- The Flask examples run **fully** using Flask's built-in test client, which drives a real application and returns real responses without needing to start a live server. FastAPI requires an ASGI server to run, so it is shown as illustrative code with explanation rather than executed here.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain the HTTP request/response cycle on the server side.
2. Describe WSGI and ASGI and how they differ.
3. Build and test a small Flask application.
4. Read a FastAPI application and explain its async and typing benefits.
5. Decide when Flask or FastAPI is the better fit.

**Prerequisites:** Tracks 1 to 6, Module 8.2, and (for async context) Track 5.

---

## Part 1 · The Server-Side Request/Response Model

Module 8.2 looked at HTTP from the **client** side. A web framework handles the **server** side: it receives an incoming request, **routes** it to the right piece of code based on the path and method, runs that code (a "view" or "handler") to produce a response, and sends the response back. The framework's job is to make this routing and response-building convenient, so you write handlers and it manages the plumbing.

In [ ]:
# A conceptual sketch of what a framework does, in plain Python.
routes = {}

def route(path):
    """A tiny decorator that registers a handler for a path (frameworks do this)."""
    def register(func):
        routes[path] = func
        return func
    return register

@route("/hello")
def hello():
    return {"message": "hello world"}

@route("/status")
def status():
    return {"ok": True}

def handle_request(path):
    """Dispatch an incoming path to the matching handler, or 404."""
    handler = routes.get(path)
    if handler is None:
        return 404, {"error": "not found"}
    return 200, handler()

print(handle_request("/hello"))
print(handle_request("/status"))
print(handle_request("/missing"))

## Part 2 · WSGI and ASGI

Python web frameworks and servers agree on a common interface so any framework works with any compatible server. **WSGI** (Web Server Gateway Interface) is the long-standing **synchronous** standard: one request is handled at a time per worker. **ASGI** (Asynchronous Server Gateway Interface) is the newer standard that also supports **asynchronous** handlers and long-lived connections such as WebSockets, building on the `async`/`await` model from Track 5.

In short: WSGI is synchronous and mature (Flask, Django historically); ASGI is asynchronous and modern (FastAPI, Starlette). The choice of framework usually determines which one you use; the server (Gunicorn for WSGI, Uvicorn for ASGI) runs your app in production.

In [ ]:
# The shape of a raw WSGI application: a callable taking environ and start_response.
def wsgi_app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    status = "200 OK"
    headers = [("Content-Type", "text/plain")]
    start_response(status, headers)
    return [f"you requested {path}".encode()]

# We can invoke it directly to see the pattern (frameworks build on this).
captured = {}
def start_response(status, headers):
    captured["status"] = status

body = wsgi_app({"PATH_INFO": "/demo"}, start_response)
print("status:", captured["status"])
print("body:", b"".join(body).decode())
print("Frameworks like Flask implement this interface so you never write it by hand.")

## Part 3 · A Flask Application (Running)

**Flask** is a small, approachable WSGI framework. You create an app, decorate functions with routes, and return responses. Normally you would run `flask run` to start a server, but Flask provides a **test client** that sends requests to the app in-process and returns the responses, which lets us run a real Flask app right here and see genuine results.

In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route("/")
def index():
    return jsonify(message="welcome")

@app.route("/greet")
def greet():
    # Read a query parameter, with a default.
    name = request.args.get("name", "world")
    return jsonify(greeting=f"Hello, {name}")

@app.route("/books/<int:book_id>")
def get_book(book_id):
    # A path parameter, typed as an int by the <int:...> converter.
    catalogue = {1: "Dune", 2: "Neuromancer"}
    title = catalogue.get(book_id)
    if title is None:
        return jsonify(error="not found"), 404      # a tuple sets the status code
    return jsonify(id=book_id, title=title)

# The test client drives the real app without starting a server.
client = app.test_client()
print("GET / ->", client.get("/").get_json())
print("GET /greet?name=Asha ->", client.get("/greet?name=Asha").get_json())
print("GET /books/1 ->", client.get("/books/1").get_json())
missing = client.get("/books/99")
print("GET /books/99 ->", missing.status_code, missing.get_json())

In [ ]:
from flask import Flask, jsonify, request

# Flask handles POST and JSON bodies as easily as GET.
app = Flask(__name__)
store = []

@app.route("/items", methods=["GET", "POST"])
def items():
    if request.method == "POST":
        data = request.get_json()
        store.append(data)
        return jsonify(created=data), 201       # 201 Created
    return jsonify(items=store)

client = app.test_client()
print("POST /items ->", client.post("/items", json={"name": "pen"}).get_json())
print("POST /items ->", client.post("/items", json={"name": "book"}).get_json())
print("GET /items  ->", client.get("/items").get_json())

## Part 4 · A FastAPI Application (Illustrative)

**FastAPI** is a modern **ASGI** framework built on type hints and `async`/`await`. You declare request and response shapes with type annotations (often using Pydantic models), and FastAPI uses them to validate input automatically, convert data, and generate interactive API documentation. It is asynchronous by default, which suits high-concurrency input/output workloads.

FastAPI needs an ASGI server (such as Uvicorn) to run, so the code below is **illustrative**, read it for the shape and the ideas rather than running it here. Notice how much the type hints do for you.

In [ ]:
# Illustrative FastAPI code (requires `uvicorn app:app` to run in a real environment).
fastapi_example = """
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# A Pydantic model declares the expected request body and validates it automatically.
class Book(BaseModel):
    title: str
    author: str
    year: int

catalogue = {}

@app.get("/books/{book_id}")          # path parameter, typed below
async def get_book(book_id: int):     # async handler; FastAPI awaits it
    if book_id not in catalogue:
        return {"error": "not found"}
    return catalogue[book_id]

@app.post("/books")
async def create_book(book: Book):    # the body is parsed and validated into a Book
    new_id = len(catalogue) + 1
    catalogue[new_id] = book.model_dump()
    return {"id": new_id, **book.model_dump()}
"""
print(fastapi_example)
print("Key ideas: type hints drive validation and docs; handlers are async by default.")

The standout features in that example: the `book_id: int` annotation makes FastAPI convert and validate the path parameter; the `Book` model means a malformed request body is rejected with a clear error before your code runs; and FastAPI automatically produces interactive documentation (a Swagger UI) from these annotations. The `async def` handlers integrate with the asyncio model from Track 5, so the framework can handle many concurrent requests efficiently.

## Part 5 · Choosing Between Them

Both frameworks are excellent; the choice depends on your needs:

- **Flask** is small, synchronous, and has a gentle learning curve and a vast ecosystem of extensions. It is a great default for traditional web apps, server-rendered pages, smaller services, and when synchronous code is fine.
- **FastAPI** is asynchronous, leans on type hints for automatic validation and documentation, and excels at high-concurrency APIs and modern services that benefit from async input/output. Its automatic docs are a productivity boon for API work.

Other options exist: **Django** is a large, batteries-included framework for full applications (admin, ORM, auth); **Starlette** is the lightweight ASGI toolkit under FastAPI. For learning and most APIs today, Flask and FastAPI are the two to know. As the scope note says, full web development is its own course; this module gives you the map.

In [ ]:
# A compact decision guide expressed as data.
guide = [
    ("Server-rendered pages / classic web app", "Flask (or Django for large apps)"),
    ("Small synchronous service or prototype",  "Flask"),
    ("High-concurrency async API",              "FastAPI"),
    ("Automatic validation and API docs",       "FastAPI"),
    ("Full-stack app with admin and auth built in", "Django"),
]
print(f"{'need':46} {'reach for'}")
for need, choice in guide:
    print(f"{need:46} {choice}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A minimal Flask route (Easy)

In [ ]:
from flask import Flask, jsonify
app = Flask(__name__)

@app.route("/ping")
def ping():
    return jsonify(pong=True)

print(app.test_client().get("/ping").get_json())

### Example 2 — Reading a query parameter (Easy)

In [ ]:
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.route("/echo")
def echo():
    return jsonify(text=request.args.get("text", ""))

print(app.test_client().get("/echo?text=hello").get_json())

### Example 3 — A typed path parameter (Medium)

In [ ]:
from flask import Flask, jsonify
app = Flask(__name__)

@app.route("/square/<int:n>")
def square(n):
    return jsonify(input=n, square=n * n)

c = app.test_client()
print(c.get("/square/7").get_json())
# A non-integer path will not match this route, yielding a 404.
print("non-integer path status:", c.get("/square/abc").status_code)

### Example 4 — Returning a custom status code (Medium)

In [ ]:
from flask import Flask, jsonify, request
app = Flask(__name__)

@app.route("/age", methods=["POST"])
def age():
    data = request.get_json()
    if data.get("age", -1) < 0:
        return jsonify(error="age must be non-negative"), 400   # Bad Request
    return jsonify(ok=True), 200

c = app.test_client()
print(c.post("/age", json={"age": 30}).status_code)
print(c.post("/age", json={"age": -5}).get_json(), c.post("/age", json={"age": -5}).status_code)

### Example 5 — A small Flask CRUD service (Difficult)

In [ ]:
from flask import Flask, jsonify, request
app = Flask(__name__)
notes = {}
next_id = {"value": 1}

@app.route("/notes", methods=["GET", "POST"])
def notes_collection():
    if request.method == "POST":
        nid = next_id["value"]; next_id["value"] += 1
        notes[nid] = request.get_json()["text"]
        return jsonify(id=nid, text=notes[nid]), 201
    return jsonify(notes=notes)

@app.route("/notes/<int:nid>", methods=["GET", "DELETE"])
def note_item(nid):
    if nid not in notes:
        return jsonify(error="not found"), 404
    if request.method == "DELETE":
        removed = notes.pop(nid)
        return jsonify(deleted=removed)
    return jsonify(id=nid, text=notes[nid])

c = app.test_client()
print("create:", c.post("/notes", json={"text": "first note"}).get_json())
print("create:", c.post("/notes", json={"text": "second note"}).get_json())
print("read:  ", c.get("/notes/1").get_json())
print("delete:", c.delete("/notes/1").get_json())
print("after: ", c.get("/notes").get_json())

### Example 6 — Reading FastAPI structure (Difficult)

In [ ]:
# Illustrative only. This shows how type hints provide validation and structure.
snippet = """
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()

class Order(BaseModel):
    item: str
    quantity: int = Field(gt=0)        # validation: quantity must be > 0

@app.post("/orders")
async def create_order(order: Order):  # body validated into an Order automatically
    total = order.quantity * 10
    return {"item": order.item, "quantity": order.quantity, "total": total}
"""
print(snippet)
print("If quantity <= 0, FastAPI rejects the request with a 422 error before the")
print("handler runs, purely from the Field(gt=0) annotation, no manual checks needed.")

---

## Exercises

The Flask exercises run with the test client, as shown above. The FastAPI exercise is a reading/explanation task.

**Exercise 1 (Easy).** Build a Flask app with a `/time` route that returns `{"service": "clock"}` as JSON, and call it with the test client.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Add a `/double` route that reads a query parameter `n` (as an int via `request.args.get(..., type=int)`) and returns `{"result": n*2}`. Test it.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Build a Flask route `/users/<int:uid>` that returns a user from a small dict, or a 404 with an error message if the id is unknown. Test both cases.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Build a `/login` POST route that returns 200 with `{"ok": True}` if the JSON body has `password == "secret"`, else 401 with an error. Test both.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Build a small Flask collection at `/tasks` supporting POST (create, returning 201 with an id) and GET (list all). Create two tasks and then list them, using the test client.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
from flask import Flask, jsonify
app = Flask(__name__)

@app.route("/time")
def time_route():
    return jsonify(service="clock")

print(app.test_client().get("/time").get_json())

**Solution 2**

In [ ]:
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.route("/double")
def double():
    n = request.args.get("n", default=0, type=int)
    return jsonify(result=n * 2)

print(app.test_client().get("/double?n=21").get_json())

**Solution 3**

In [ ]:
from flask import Flask, jsonify
app = Flask(__name__)
users = {1: "Asha", 2: "Ben"}

@app.route("/users/<int:uid>")
def get_user(uid):
    if uid not in users:
        return jsonify(error="not found"), 404
    return jsonify(id=uid, name=users[uid])

c = app.test_client()
print(c.get("/users/1").get_json())
print(c.get("/users/9").status_code, c.get("/users/9").get_json())

**Solution 4**

In [ ]:
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.route("/login", methods=["POST"])
def login():
    if request.get_json().get("password") == "secret":
        return jsonify(ok=True), 200
    return jsonify(error="unauthorized"), 401

c = app.test_client()
print(c.post("/login", json={"password": "secret"}).status_code)
print(c.post("/login", json={"password": "wrong"}).status_code)

**Solution 5**

In [ ]:
from flask import Flask, request, jsonify
app = Flask(__name__)
tasks = {}
counter = {"n": 0}

@app.route("/tasks", methods=["GET", "POST"])
def tasks_route():
    if request.method == "POST":
        counter["n"] += 1
        tasks[counter["n"]] = request.get_json()["title"]
        return jsonify(id=counter["n"], title=tasks[counter["n"]]), 201
    return jsonify(tasks=tasks)

c = app.test_client()
print(c.post("/tasks", json={"title": "a"}).get_json())
print(c.post("/tasks", json={"title": "b"}).get_json())
print(c.get("/tasks").get_json())

---

## Summary and Key Points

- A web framework handles the server side of HTTP: it routes an incoming request (by path and method) to a handler, runs it to build a response, and sends it back, managing the plumbing for you.
- WSGI is the synchronous standard (Flask, classic Django); ASGI is the asynchronous standard (FastAPI, Starlette) supporting async handlers and WebSockets; production servers are Gunicorn (WSGI) and Uvicorn (ASGI).
- Flask is a small synchronous framework: decorate functions with routes, return JSON or tuples with status codes; its test client drives the real app in-process for easy testing.
- FastAPI is an asynchronous, type-hint-driven framework: annotations and Pydantic models validate requests automatically and generate interactive docs, with async handlers for high concurrency.
- Choose Flask for synchronous apps, prototypes, and server-rendered pages (Django for large full-stack apps); choose FastAPI for high-concurrency async APIs and automatic validation and documentation. Full web development is a course in itself.

### Next module: 8.5 — Orientation: The Data Stack

The final module of this track orients you to scientific Python: NumPy arrays, Pandas DataFrames, and quick plotting with matplotlib, as a launchpad into data analysis.